In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data, check_unifac_parameters, check_pure_gc_parameters
from src.mixture.unifac.pyomo_vle import pyo_vle
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# step1: define the compounds
mixture = {'acetone': {'CH3':1, 'CH3CO':1},
           #'methyl_acetate': {'CH3': 1, 'CH3COO': 1},
           #'methanol': {'CH3OH':1},
           'n-heptane': {'CH3':2, 'CH2':5},
           #'water': {'H2O':1}
           #'DOH': {'DOH':1}
           
           
           #'E': {'CH2':1, 'HCO':2}
           #'E': {'CH3':1, 'CH2':2, 'CH3CO':1, 'CH2O':1}
           #'E': {'CH3':3, 'CH':1, 'CH2COO':1}
           }
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)
# load unifac data
rk_data, qk_data, interaction_data = load_unifac_data()

In [ ]:
# check if unifac dat are available
check_unifac_parameters(mixture)

In [ ]:
compounds = list(mixture.keys())
all_groups = set()
for compound in mixture.values():
    all_groups.update(compound.keys())
print("All groups:", all_groups)
print("Groups:", all_groups)
print("\n=== Rk parameters ===")
for group in all_groups:
    print(f"{group}: {rk_data.get(group, 'MISSING')}")

print("\n=== Qk parameters ===")
for group in all_groups:
    print(f"{group}: {qk_data.get(group, 'MISSING')}")

print("\n=== Interaction parameters (a_ij) ===")
for g1 in all_groups:
    for g2 in all_groups:
        a_val = interaction_data.get(g1, {}).get(g2, 'MISSING')
        print(f"a[{g1}, {g2}] = {a_val}")

In [ ]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
check_pure_gc_parameters(mixture, properties=prop_list)

In [ ]:
#caculate the properties
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)
# fix the properties of acetone
pure_props['acetone']['tc'] = 508.1
pure_props['acetone']['pc'] = 48.0
pure_props['acetone']['tb'] = 329.3
pure_props['acetone']['hv'] = 31.27

# pure_props['methyl_acetate']['tc'] = 510.0     # K
# pure_props['methyl_acetate']['pc'] = 47.5      # bar
# pure_props['methyl_acetate']['tb'] = 330.0

pure_props['n-heptane']['tc'] = 540.0
pure_props['n-heptane']['pc'] = 27.4
pure_props['n-heptane']['tb'] = 371.5
pure_props['n-heptane']['hv'] = 36.3
pure_props

In [ ]:
# step3: calculate the Antoine coefficients
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props, method='enthalpy_vap')
    print('optimization results was', results['termination_condition'] )
    antoine_params[compound] = {}
    # antoine_params_ini[compound]['A'] = results['A0']
    # antoine_params_ini[compound]['B'] = results['B0']
    # antoine_params_ini[compound]['C'] = results['C0']
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']
print(antoine_params)

In [ ]:
# step4: perform the vle

# define number of data points
n_data = 100
# set composition
# generate all binary combinations for ternary system
# x1-x2 binary (x3=0)
x0 = 0.00
xf = 1.00
x1_12 = np.linspace(x0, xf, n_data)
x2_12 = 1 - x1_12
if len(mixture.keys()) == 3:
    x3_12 = np.zeros(n_data)

if len(mixture.keys()) == 3:
    # x1-x3 binary (x2=0)
    x1_13 = np.linspace(x0, xf, n_data)
    x2_13 = np.zeros(n_data)
    x3_13 = 1 - x1_13

    # x2-x3 binary (x1=0)
    x1_23 = np.zeros(n_data)
    x2_23 = np.linspace(x0, xf, n_data)
    x3_23 = 1 - x2_23

# combine all binary combinations
if len(mixture.keys()) == 2:
    x = np.column_stack([x1_12, x2_12])
elif len(mixture.keys()) == 3:
    x = np.vstack([
        np.column_stack([x1_12, x2_12, x3_12]),
        np.column_stack([x1_13, x2_13, x3_13]),
        np.column_stack([x1_23, x2_23, x3_23])
    ])
# provide nitial guess for temperature

# Extract boiling points for each compound
tb_values = [pure_props[compound]['tb'] for compound in mixture.keys()]
if len(mixture.keys()) == 3:
    tb1, tb2, tb3 = tb_values  # assuming 3 components
elif len(mixture.keys()) == 2:
    tb1, tb2 = tb_values  # assuming 2 components
# Create T0 for each binary combination
# x1-x2 binary: T ranges from tb2 (x1=0, x2=1) to tb1 (x1=1, x2=0)
T0_12 = np.linspace(tb2, tb1, n_data)# np.ones(n_data)*(tb1+tb2)/2#

if len(mixture.keys()) == 3:
    # x1-x3 binary: T ranges from tb3 (x1=0, x3=1) to tb1 (x1=1, x3=0)
    T0_13 = np.linspace(tb3, tb1, n_data)

    # x2-x3 binary: T ranges from tb3 (x2=0, x3=1) to tb2 (x2=1, x3=0)
    T0_23 = np.linspace(tb3, tb2, n_data)

# Combine all T0 values
if len(mixture.keys()) == 3:
    T0 = np.concatenate([T0_12, T0_13, T0_23])
elif len(mixture.keys()) == 2:
    T0 = T0_12


In [ ]:
# plot satuyrated pressures
Psat = np.ones_like(x)
for i, com in enumerate(mixture):
    Psat[:,i] = antoine_calc.antoine_correlation(A=antoine_params[com]['A'],
                                                  B=antoine_params[com]['B'],
                                                  C=antoine_params[com]['C'],
                                                  T=T0)

print(f" the mimium Psat is {Psat.min()} bar and the maximum Psat is {Psat.max()} bar")
print(f" the the psat at the boiling points are: {Psat[0,1]} bar and {Psat[-1,0]} bar")


# plot the psat and highlight Pc,Tc and Tb
plt.figure()
for i, com in enumerate(mixture):
    color = f'C{i}'  # Use matplotlib's default color cycle
    plt.plot(T0, Psat[:,i], color=color, label=f'Psat {com}')
    plt.scatter(pure_props[com]['tb'], 1.01325, color=color, marker='o', label=f'Tb, P0 of {com}') 
    # make a hotizontal like at P=1.01325 bar
    #plt.scatter(pure_props[com]['tc'], pure_props[com]['pc'], color=color, marker='x', label=f'Tc, Pc of {com}')  
plt.hlines(y=1.01325, xmin=min(T0), xmax=max(T0), color='k', linestyle='--', label=f'P0 = 1.01325 bar')
plt.xlabel('Temperature (K)')
plt.ylim(0,)
plt.ylabel('Psat (bar)')
plt.legend()
plt.title('Vapor Pressure vs Temperature')
plt.show()

In [ ]:
from src.mixture.unifac.interface import unifac
init_test = unifac(mixture, x, T0, rk_data, qk_data, interaction_data)
print("Gamma values:", init_test['gamma'])
print("Any NaN or inf?", np.any(np.isnan(init_test['gamma'])) or np.any(np.isinf(init_test['gamma'])))
print("Min gamma:", np.min(init_test['gamma']))
print("Max gamma:", np.max(init_test['gamma']))
print("Gamma range:", init_test['gamma'])

In [ ]:
# perform the VLE calculation
results, model = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=True)


In [ ]:
# step5: lets plot the results in T-xy diagram and inspect the gamma values
import matplotlib.pyplot as plt

# Extract results
# retrieving the solutions
gamma_pyo = np.empty_like(x)
T_pyo = np.empty_like(T0)
y_pyo = np.empty_like(x)
psat_pyo = np.empty_like(x)

for i in model.gamma.keys():
    gamma_pyo[i] = model.gamma[i].value

for i in model.T.keys():
    T_pyo[i] = model.T[i].value

for i in model.y.keys():
    y_pyo[i] = model.y[i].value

for i in model.psat.keys():
    psat_pyo[i] = model.psat[i].value

# Create T-xy plots for all three binary combinations
if len(mixture.keys()) == 3:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
elif len(mixture.keys()) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Component 1-2 diagram (first 100 points)
axes[0].plot(y_pyo[:n_data, 0], T_pyo[:n_data], 'b-', label='Vapor')
axes[0].plot(x[:n_data, 0], T_pyo[:n_data], 'r-', label='Liquid')
axes[0].set_ylabel('Temperature (K)')
axes[0].set_xlabel('Component 1 Fraction')
axes[0].set_title('T-xy Diagram: Component 1-2')
axes[0].legend()
axes[0].grid()

if len(mixture.keys()) == 2:
    axes[1].plot(x[:n_data, 0], y_pyo[:n_data, 0], 'b-', label='vapour')
    axes[1].plot(x[:n_data, 0], x[:n_data, 0], 'r-', label='x=y')
    axes[1].set_ylabel('Temperature (K)')
    axes[1].set_xlabel('Component 1 Fraction')
    axes[1].set_title('T-xy Diagram: Component 1-2')
    axes[1].legend()
    axes[1].grid()

if len(mixture.keys()) == 3:
    # Component 1-3 diagram (points 100-200)
    axes[1].plot(y_pyo[n_data:2*n_data, 0], T_pyo[n_data:2*n_data], 'b-', label='Vapor')
    axes[1].plot(x[n_data:2*n_data, 0], T_pyo[n_data:2*n_data], 'r-', label='Liquid')
    axes[1].set_ylabel('Temperature (K)')
    axes[1].set_xlabel('Component 1 Fraction')
    axes[1].set_title('T-xy Diagram: Component 1-3')
    axes[1].legend()
    axes[1].grid()

    # Component 2-3 diagram (points 200-300)
    axes[2].plot(y_pyo[2*n_data:3*n_data, 1], T_pyo[2*n_data:3*n_data], 'b-', label='Vapor')
    axes[2].plot(x[2*n_data:3*n_data, 1], T_pyo[2*n_data:3*n_data], 'r-', label='Liquid')
    axes[2].set_ylabel('Temperature (K)')
    axes[2].set_xlabel('Component 2 Fraction')
    axes[2].set_title('T-xy Diagram: Component 2-3')
    axes[2].legend()
    axes[2].grid()#

plt.tight_layout()
plt.show()

# 

In [ ]:
# calculate miscibility using unifac interface
from src.mixture.unifac.interface import unifac
unifac_results = unifac(mixture, x, np.ones_like(T0)*tb1, rk_data, qk_data, interaction_data)
gamma_unifac = unifac_results['gamma']
# Calculate Gibbs mixing energy (dG/RT)
G_mix = (np.sum(x * np.log(gamma_unifac), axis=1) +
         np.sum(np.ma.log(x).filled(0) * x, axis=1))
G_mix

In [ ]:
G2_mix = np.diff(np.diff(G_mix))
G2_mix

In [ ]:
# Check miscibility criteria:
# 1. G_mix <= 0 (thermodynamic stability)
# 2. d2G >= 0 (mechanical stability)
is_miscible = np.all(G_mix[1:-1] <= 0) and np.all(G2_mix >= 0)
is_miscible

In [ ]:
# check for azeotrope using algorithm
from src.mixture.unifac.pyomo_azeo import pyo_azeo
T0_az = max(tb1,tb2)#np.array((T_min+T_max)/2)
x0_az = np.array([[0.5, 0.5]])
results_az, model_az = pyo_azeo(x0_az, T0_az, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)
solv_stat = str(results_az.solver.status)
term_cond = str(results_az.solver.termination_condition)
if solv_stat != 'ok' or term_cond != 'optimal':
    print(f"No azeotrope detected")
    print(f"         Solver status: {solv_stat}, Termination: {term_cond}")
else:
    x_az = np.array([model_az.x[i].value for i in model_az.x.keys()])
    T_az = np.array([model_az.T[i].value for i in model_az.T.keys()])
    y_az = np.array([model_az.y[i].value for i in model_az.y.keys()])
    print(f"Azeotrope detected at T={T_az} K and x={x_az} and y={y_az}")

In [ ]:
T_az

In [ ]:
results_az

Now lets check for the mscibility between water and the two other components according to the two relations:

\begin{equation*}
\frac{\Delta G_{\text{mix}}}{RT} = \sum_i x_i \log(\gamma_i) + \sum_i x_i \log(x_i) \leq 0
\end{equation*}

\begin{equation*}
  \frac{\partial^2 \left(\frac{\Delta G^{mix}}{RT}\right)}{\partial x_i^2} = \frac{\Delta G^{mix}}{RT}\bigg|_{x_{p+1,i}} - 2 \frac{\Delta
  G^{mix}}{RT}\bigg|_{x_{p,i}} + \frac{\Delta G^{mix}}{RT}\bigg|_{x_{p-1,i}} \geq 0
\end{equation*}

In [ ]:
# miscibility between acetone and water
Gmix = (np.sum(x * np.log(gamma_pyo), axis=1) +
                np.sum(np.ma.log(x).filled(0) * x, axis=1))
Gmix_13 = Gmix[n_data:2*n_data]
Gmix_23 = Gmix[2*n_data:]
# calculate 2nd derivatives
Gmix_13_2nd = np.diff(np.diff(Gmix_13))
Gmix_23_2nd = np.diff(np.diff(Gmix_23))
# miscibility between methanol and water
print(f"mixture of acetone and E is miscible: {np.all(Gmix_13 <= 0) and np.all(Gmix_13_2nd >= 0)}")
print(f"mixture of methanol and E is miscible: {np.all(Gmix_23 <= 0) and np.all(Gmix_23_2nd >= 0)}")

In [ ]:
Gmix_13

In [ ]:
Gmix_13[1:-1]

In [ ]:
Gmix_13_2nd

A final element is to caulcate the relative volatility, selecivity and capacity at infinite dilution

\begin{equation*}
\alpha_{A,B}^{\infty} = \left(\frac{K_A^{\infty}}{K_B^{\infty}}\right)_E = \left(\frac{\gamma_A^{\infty}}{P^*_A} * \frac{\gamma_B^{\infty}}{P^*_B} \right)_E  
\end{equation*}


\begin{equation*}
S_{A,B}^{\infty} = \left(\frac{\gamma_A^{\infty}}{\gamma_B^{\infty}}\right)_E
\end{equation*}

\begin{equation*}
C_{B,E}^{\infty} = \left(\frac{1}{\gamma_B^{\infty}}\right)_E
\end{equation*}

\begin{equation*}
C_{B,E}^{\infty} * S_{A,B}^{\infty}
\end{equation*}

In [ ]:
x[[0, n_data-1,n_data]]

In [ ]:
# calculate the relative volatility
alpha = gamma_pyo[n_data,0]/psat_pyo[n_data,0] * gamma_pyo[n_data,1]/gamma_pyo[n_data,1]
alpha

In [ ]:
# calculate selectivity
S = gamma_pyo[n_data,0]/gamma_pyo[n_data,1]
S

In [ ]:
# calculate the capacity
C = 1/gamma_pyo[n_data,1]
C

In [ ]:
# calculating the product
S*C